<a href="https://colab.research.google.com/github/Balara00/ML_Lab01/blob/main/label_1/ML_Lab01_label_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import required packages and libraries

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#constants
L1 = 'label_1'
L2 = 'label_2'
L3 = 'label_3'
L4 = 'label_4'

LABELS = [L1, L2, L3, L4]
AGE_LABEL = L2
FEATURES = [f"feature_{i}" for i in range (1,257)]

Importing dataset

In [ ]:
from google.colab import drive
MOUNT_PATH='/content/drive'
drive.mount(MOUNT_PATH)

In [ ]:
WORKING_DIR=f"{MOUNT_PATH}/MyDrive/ML_Lab 1"

In [ ]:
train = pd.read_csv(f"{WORKING_DIR}/train.csv")
train.head()

In [ ]:
valid = pd.read_csv(f"{WORKING_DIR}/valid.csv")
valid.head()

In [ ]:
test = pd.read_csv(f"{WORKING_DIR}/test.csv")
test.head()

In [ ]:
train[LABELS + [FEATURES[i] for i in range(0, 256, 32)]].describe()
train.info()

Scaling dataset

In [ ]:
from sklearn.preprocessing import StandardScaler

x_train = {}
y_train = {}
x_valid = {}
y_valid = {}
x_test = {}
y_test = {}

for target_label in LABELS:
  train_ds = train[train['label_2'].notna()] if target_label == 'label_2' else train
  valid_ds = valid
  test_ds = test

  scaler = StandardScaler()
  x_train[target_label] = pd.DataFrame(scaler.fit_transform(train_ds.drop(LABELS, axis = 1)), columns=FEATURES)
  y_train[target_label] = train_ds[target_label]

  x_valid[target_label] = pd.DataFrame(scaler.transform(valid_ds.drop(LABELS, axis = 1)), columns=FEATURES)
  y_valid[target_label] = valid_ds[target_label]

  x_test[target_label] = pd.DataFrame(scaler.transform(test_ds.drop(LABELS, axis = 1)), columns=FEATURES)
  y_test[target_label] = test_ds[target_label]

In [ ]:
y_train['label_2']

In [ ]:
from sklearn import svm

clf = svm.SVC(kernel = 'linear')
clf.fit(x_train[L1], y_train[L1])

In [ ]:
from sklearn import metrics

y_pred = clf.predict(x_valid[L1])
y_pred_test_before = clf.predict(x_test[L1])

In [ ]:
print('Predicted labels before feature engineering:', y_pred_test_before)

In [ ]:
print (metrics.confusion_matrix(y_valid[L1], y_pred))
print (metrics.accuracy_score(y_valid[L1], y_pred))
print (metrics.precision_score(y_valid[L1], y_pred, average="weighted"))
print (metrics.recall_score(y_valid[L1], y_pred, average="weighted"))

# Feature Engineering

### SelectKBest

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(f_classif, k=100)
x_train_new = selector.fit_transform(x_train[L1], y_train[L1])
print("Shape: ", x_train_new.shape)

In [ ]:
clf = svm.SVC(kernel = 'linear')
clf.fit(x_train_new, y_train[L1])

In [ ]:
y_pred = clf.predict(selector.transform(x_valid[L1]))

print (metrics.confusion_matrix(y_valid[L1], y_pred))
print (metrics.accuracy_score(y_valid[L1], y_pred))
print (metrics.precision_score(y_valid[L1], y_pred, average="weighted"))
print (metrics.recall_score(y_valid[L1], y_pred, average="weighted"))

### PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, svd_solver='full')
pca.fit(x_train[L1])
x_train_trf = pd.DataFrame(pca.transform(x_train[L1]))
x_valid_trf = pd.DataFrame(pca.transform(x_valid[L1]))

print("Shape after PCA: ", x_train_trf.shape)

In [ ]:
clf = svm.SVC(kernel = 'linear')
clf.fit(x_train_trf, y_train[L1])

In [ ]:
y_pred = clf.predict(x_valid_trf)

print (metrics.confusion_matrix(y_valid[L1], y_pred))
print (metrics.accuracy_score(y_valid[L1], y_pred))
print (metrics.precision_score(y_valid[L1], y_pred, average="weighted"))
print (metrics.recall_score(y_valid[L1], y_pred, average="weighted"))

### PCA with SelectKBest

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

# Step 1: Feature selection with SelectKBest
selector = SelectKBest(f_classif, k=120)
x_train_new = selector.fit_transform(x_train[L1], y_train[L1])

# Step 2: Apply PCA to the selected features
pca = PCA(n_components=0.95, svd_solver='full')
x_train_pca = pca.fit_transform(x_train_new)

print("Shape after PCA: ", x_train_pca.shape)

In [ ]:
# Create a validation set with the same feature transformations
x_valid_new = selector.transform(x_valid[L1])
x_valid_pca = pca.transform(x_valid_new)

In [ ]:
x_test_new = selector.transform(x_test[L1])
x_test_pca = pca.transform(x_test_new)

In [ ]:
# Train a classifier (SVM) on the PCA-transformed features
clf = svm.SVC(kernel='linear')
clf.fit(x_train_pca, y_train[L1])

In [ ]:
# Make predictions on the validation set
y_pred = clf.predict(x_valid_pca)

# Evaluate the classifier's performance
print(metrics.confusion_matrix(y_valid[L1], y_pred))
print(metrics.accuracy_score(y_valid[L1], y_pred))
print(metrics.precision_score(y_valid[L1], y_pred, average="weighted"))
print(metrics.recall_score(y_valid[L1], y_pred, average="weighted"))

In [ ]:
y_pred_test_after = clf.predict(x_test_pca)
print('Predicted labels after feature engineering:', y_pred_test_after)

**Output**

In [ ]:
# output_df = pd.DataFrame({
#     'Predicted labels before feature engineering': y_pred_test_before,
#     'Predicted labels after feature engineering': y_pred_test_after,
#     'No. of new features': x_test_pca.shape[1]
# })

# # for i in range(x_test_pca.shape[1]):
# #   output_df[f'new_feature_{i+1}'] = x_test_pca[:, i]

# for i in range(257):  # Looping from 0 to 256 inclusive
#     column_name = f'new_feature_{i+1}'  # Construct the column name

#     # Check if the feature exists in x_test_pca
#     if i < x_test_pca.shape[1]:
#         output_df[column_name] = x_test_pca[:, i]  # Fill with the feature data
#     else:
#         output_df[column_name] = None  # Fill with blank or NaN values


# output_df.head()

# Create the base DataFrame
output_df = pd.DataFrame({
    'Predicted labels before feature engineering': y_pred_test_before,
    'Predicted labels after feature engineering': y_pred_test_after,
    'No of new features': [x_test_pca.shape[1]] * len(y_pred_test_before)
})

# List to store all the new feature Series
new_features = []

# Extract new features from x_test_pca
for i in range(min(256, x_test_pca.shape[1])):
    column_name = f'new_feature_{i+1}'
    new_features.append(pd.Series(x_test_pca[:, i], name=column_name))

# If there are any remaining columns to reach 256, fill them with NaN
for i in range(x_test_pca.shape[1], 256):
    column_name = f'new_feature_{i+1}'
    new_features.append(pd.Series([None] * len(output_df), name=column_name))

# Concatenate all the new feature columns to output_df
output_df = pd.concat([output_df] + new_features, axis=1)

output_df.head()


In [ ]:
# Save the DataFrame to the specified CSV file path
output_df.to_csv(f"{WORKING_DIR}/190572L_label_1_final.csv", index=False)


## Co-relation Matrix

In [ ]:
# Convert the PCA-transformed features (x_train_pca) to a Pandas DataFrame
x_train_pca_df = pd.DataFrame(x_train_pca)

# Calculate the correlation matrix
corr_matrix = x_train_pca_df.corr()
corr_matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr_treshold = 0.5
filterred_correlation_matrix = corr_matrix[(corr_matrix > corr_treshold) | (corr_matrix < -corr_treshold)]
plt.figure(figsize=(10,8))
sns.heatmap(filterred_correlation_matrix, annot=True, cmap='coolwarm', center = 0)